In [2]:
import import_ipynb
import documents  

File '../data/text_files/python_intro.txt' created successfully.
File '../data/text_files/python_advanced.txt' created successfully.


d:\ProjectsGit\ai-ml\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 7/7 [00:01<00:00,  6.95it/s]


DOCUMENT INGESTION
Normal documents: 2
PDF documents: 128
Total documents: 130
1. ..\data\text_files\python_advanced.txt | 1537 characters
2. ..\data\text_files\python_intro.txt | 3448 characters
3. ..\data\pdf\Attention Mechanisms in Computer Vision.pdf | 4650 characters
4. ..\data\pdf\Attention Mechanisms in Computer Vision.pdf | 4491 characters
5. ..\data\pdf\Attention Mechanisms in Computer Vision.pdf | 4434 characters
6. ..\data\pdf\Attention Mechanisms in Computer Vision.pdf | 2164 characters
7. ..\data\pdf\Attention Mechanisms in Computer Vision.pdf | 5778 characters
8. ..\data\pdf\Attention Mechanisms in Computer Vision.pdf | 5293 characters
9. ..\data\pdf\Attention Mechanisms in Computer Vision.pdf | 3841 characters
10. ..\data\pdf\Attention Mechanisms in Computer Vision.pdf | 5605 characters
11. ..\data\pdf\Attention Mechanisms in Computer Vision.pdf | 4674 characters
12. ..\data\pdf\Attention Mechanisms in Computer Vision.pdf | 3858 characters
13. ..\data\pdf\Attention Mecha

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2957.86it/s]


Model all-MiniLM-L6-v2 loaded successfully.
Embedding dimension: 384
Vector store initialized at ../data/vector_store
Collection: 'pdf_documents'
Existing documents: 0
Generating embeddings for 628 texts.


...

shape of embeddings: (628, 384)
Added 628 documents to the vector store.
Collection now contains 628 documents.
Retrieving documents for query: 'What is Python programming language?'
Top K: 5
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 125.79it/s]


shape of embeddings: (1, 384)
Query embedding shape: (384,)
ChromaDB query completed.
Retrieved 5 documents.
Retrieving documents for query: 'Hochreiter and Schmidhuber'
Top K: 5
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 71.22it/s]

shape of embeddings: (1, 384)
Query embedding shape: (384,)
ChromaDB query completed.
Retrieved 5 documents.


### Intrigation of vectorDB with llm pipeline


In [11]:
## simple rag pipeline using groq api and langchain
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()


###initilize the groq llm
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key, model="openai/gpt-oss-120b", temperature=0.1, max_tokens=1024)

#simple rag function: retiver context + genrate response
def rag_simple(query, retriever,llm,top_k=3):
    #retrive context from the retriver
    results = retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else "No context found."
    if not context:
        return "No relevent context found."

##prompt for the llm to generate response based on the query and context
    prompts=f"""use the following context to answer the question concisely and accurately:
    Context:{context}
    Question: {query}
    Answer:
    """

    response = llm.invoke(prompts)
    return response.content

In [12]:

rag_retriever = documents.rag_retriever

answer = rag_simple(
    "What is this algorithm 1: Computing ROI features by pooling from multiple feature maps?",
    rag_retriever,
    llm
)
print(answer)

Retrieving documents for query: 'What is this algorithm 1: Computing ROI features by pooling from multiple feature maps?'
Top K: 3
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 66.43it/s]

shape of embeddings: (1, 384)
Query embedding shape: (384,)
ChromaDB query completed.
Retrieved 3 documents.


**Algorithm 1** is a routine that builds a feature vector for every Region‑of‑Interest (ROI) by repeatedly aggregating point‑wise information from all backbone feature maps.

1. **Inputs** – the four multi‑scale feature maps \(F_{1\ldots4}\) produced by the backbone and a set of ROIs \(R=\{(x_r,y_r,z_r,d_{xr},d_{yr},d_{zr},\theta_r),c_{lsr}\}_{r=1}^M\) generated by the RPN.  

2. **Initialization** – each ROI \(r\) is assigned a learnable embedding \(\mathbf{r}_r\leftarrow\Theta\); the collection of these embeddings forms the initial ROI‑feature set \(RF\).

3. **Iterative pooling (repeated \(N\) times)**  
   * For each scale \(i\) from the **largest** feature map (i = 4) down to the smallest (i = 1):  
     * For every ROI \(r\):  
       * **Pool** point‑wise features \(\{(p_{rj},f_j)\}\) from the current feature map \(F_i\) that fall inside the ROI (the operation is denoted `Pool(F_i)`).  
       * **Update** the ROI embedding by applying the vector‑attention formula (Eq. 4):  
   

### Advance rag retiver pipleine


In [19]:
def advance_retriver(query, retriever, llm, top_k=5, min_score=0.5,
                         return_context=False):
        results = retriever.retrieve(
            query,
            top_k=top_k,
            score_threshold=min_score
        )

        if not results:
            output = {
                "answer": "No relevant context found.",
                "context": None,
                "query": query,
                "sources": [],
                "confident": 0.0,
            }
            return output

        context = "\n\n".join(doc["content"] for doc in results)

        sources = [{
            "source": doc.get("metadata", {}).get(
                "source_file",
                doc.get("metadata", {}).get("source", "unknown")
            ),
            "page": doc.get("metadata", {}).get("page", "unknown"),
            "score": doc.get("similarity_score", 0.0),
            "preview": doc["content"][:300] + "..."
        } for doc in results]

        confidence = max(doc.get("similarity_score", 0.0) for doc in results)

        prompt = f"""Use the following context to answer the question concisely and accurately:
    Context: {context}
    Question: {query}
    Answer:"""

        response = llm.invoke(prompt)

        output = {
            "answer": response.content,
            "sources": sources,
            "confident": confidence,
            "query": query
        }

        if return_context:
            output["context"] = context

        return output

In [21]:
## example
result=advance_retriver("DETR architecture",rag_retriever,llm,top_k=3,min_score=0.1,return_context=True)
print("answer: ",result['answer'])
print('Sources: ', result['sources'])
print('Confidence: ', result['confident'])
print('Context preview: ', result['context'][:300] + '...')


Retrieving documents for query: 'DETR architecture'
Top K: 3
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.15it/s]

shape of embeddings: (1, 384)
Query embedding shape: (384,)
ChromaDB query completed.
Retrieved 3 documents.


answer:  **DETR (DEtection TRansformer) architecture**

1. **CNN backbone** – extracts a dense feature map from the input image.  
2. **Transformer encoder‑decoder** –  
   * **Encoder**: processes the backbone features with multi‑head self‑attention, adding spatial positional encodings at every layer.  
   * **Decoder**: receives a fixed set of learned *object queries* (initially zero) together with the encoder memory. Through stacked self‑attention and encoder‑decoder attention layers (with positional encodings) it produces a set of embeddings, one per query.  
3. **Feed‑forward prediction head** – a simple FFN applied to each decoder output predicts:  
   * a class probability distribution (including a “no‑object” class), and  
   * a bounding‑box regression (using a combination of L1 and generalized‑IoU loss).  

The whole model is trained end‑to‑end with a **set‑based bipartite matching loss**, which removes duplicate predictions and eliminates the need for post‑processing steps s

### Advance rag pipeline with streaming ad all

In [29]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

def _get_score(r):
    """Extract score from dict regardless of key name."""
    for key in ('score', 'similarity_score', 'relevance_score', 'distance'):
        if key in r:
            return r[key]
    return 0.0

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2,
              stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        results = self.retriever.retrieve(question)
        results = [r for r in results if _get_score(r) >= min_score][:top_k]

        if not results:
            return {'question': question, 'answer': 'No relevant context found.',
                    'sources': [], 'summary': None, 'history': self.history}

        # Build numbered context
        context = ""
        sources = []
        for i, r in enumerate(results):
            src = r['metadata'].get('source_file', r['metadata'].get('source', 'unknown'))
            pg = r['metadata'].get('page', 'unknown')
            context += f"[{i+1}] (Source: {src}, Page: {pg})\n{r['content']}\n\n"
            sources.append({
                'source': src,
                'page': pg,
                'score': round(_get_score(r), 3),
                'preview': r['content'][:120] + '...'
            })

        # Build messages with history
        messages = [
            SystemMessage(content="Answer concisely using ONLY the provided context. Cite claims as [1], [2], etc.")
        ]
        for turn in self.history[-4:]:
            messages.append(HumanMessage(content=turn['question']))
            messages.append(AIMessage(content=turn['answer']))
        messages.append(HumanMessage(content=f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"))

        # Generate (stream or not)
        if stream:
            print("Streaming:", end="")
            parts = []
            for chunk in self.llm.stream(messages):
                if chunk.content:
                    print(chunk.content, end="", flush=True)
                    parts.append(chunk.content)
            print()
            answer = "".join(parts)
        else:
            answer = self.llm.invoke(messages).content

        # Optional summary
        summary = None
        if summarize:
            summary = self.llm.invoke(
                f"Summarize this in 2 sentences:\n{answer}"
            ).content

        # Store history
        self.history.append({'question': question, 'answer': answer,
                             'sources': sources, 'summary': summary})

        return {'question': question, 'answer': answer, 'sources': sources,
                'summary': summary, 'history': self.history}   

In [32]:
# --- Example Usage ---
advance_rag_pipeline = AdvancedRAGPipeline(rag_retriever, llm)

result = advance_rag_pipeline.query("DETR architecture in 500 words", top_k=10, min_score=0.1, summarize=True)
print("Answer:", result['answer'])
print("\nSources:", result['sources'])
print("\nSummary:", result['summary'])
    
print("\nHistory:", result['history'])

# Follow-up (history carries forward automatically)
result2 = advance_rag_pipeline.query("How does it compare to Faster R-CNN? in 500 words", top_k=10, min_score=0.1)
print("\n--- Follow-up ---")
print("Answer:", result2['answer'])   

Retrieving documents for query: 'DETR architecture in 500 words'
Top K: 5
Generating embeddings for 1 texts.


Batches: 100%|██████████| 1/1 [00:00<00:00, 80.54it/s]

shape of embeddings: (1, 384)
Query embedding shape: (384,)
ChromaDB query completed.
Retrieved 5 documents.


Answer: DETR (DEtection TRansformer) combines a convolutional backbone with a full‑transformer encoder‑decoder to produce a set‑based prediction of object classes and bounding boxes, eliminating the need for post‑processing such as non‑maximum suppression.

**Backbone and encoder** – An image is first processed by a CNN backbone that extracts dense feature maps. These image features are fed into a transformer encoder together with a spatial positional encoding. The positional encoding is added to the queries and keys at every multi‑head self‑attention layer, allowing the encoder to retain spatial information despite the permutation‑invariant nature of self‑attention [1].

**Decoder and object queries** – The decoder receives three inputs: (1) a set of learnable “object queries” (initially set to zero), (2) an output positional encoding, and (3) the memory produced by the encoder. Each decoder layer performs multi‑head self‑attention over the object queries, followed by multi‑head atten

Batches: 100%|██████████| 1/1 [00:00<00:00, 46.40it/s]

shape of embeddings: (1, 384)
Query embedding shape: (384,)
ChromaDB query completed.
Retrieved 5 documents.



--- Follow-up ---
Answer: Faster R‑CNN improves on earlier two‑stage detectors (e.g., Fast R‑CNN with Selective Search or EdgeBoxes) by replacing the external proposal generator with a Region Proposal Network (RPN) that is trained jointly with the detection network and shares convolutional features. This architectural change yields both higher accuracy and substantially faster inference.

**Accuracy gains** – When the same ZF backbone is used, Fast R‑CNN with Selective Search (≈2 000 proposals) attains 58.7 % mAP, while EdgeBoxes gives 58.6 % mAP [3]. Adding an RPN and limiting the number of proposals to ≤300 raises the mAP to 59.9 % under the Fast R‑CNN framework, already surpassing the other proposal methods despite using far fewer regions [3][1]. With a deeper ResNet‑101 backbone, Faster R‑CNN’s mAP on COCO rises from 41.5 % (VGG‑16) to 48.4 % (ResNet‑101), a 6.9‑point improvement that demonstrates the detector’s strong dependence on high‑quality features learned by the RPN [2]. Fu